# Explainability (XAI) for Perception: Methods, Sanity Checks and Limits as Safety Evidence

**Module 06 · AI Safety (ISO/PAS 8800)** · Notebook 6 of 7

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/06_AI_Safety_ISO_PAS_8800/06_explainability.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~75 min | [03 MC dropout and ensembles](03_mc_dropout_and_deep_ensembles.ipynb); CNN basics | ISO/PAS 8800:2024, ISO/IEC TR 5469:2024 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

- Understand where explainability supports safety work: debugging, safety analysis of AI errors, incident investigation and communication; and where it does not.
- Implement **gradient saliency**, **Grad-CAM**, **integrated gradients**, **occlusion sensitivity** and **Shapley values** from scratch on a small CNN, and state precisely what each one computes.
- Check the **additivity** property of Shapley values ($f(x) = \phi_0 + \sum_i\phi_i$) and the **completeness** property of integrated gradients numerically.
- Run the **model-randomisation sanity check** (Adebayo et al., 2018) and a localisation check against known object masks.
- Use XAI to uncover a **shortcut** (a spurious feature) that a test-set accuracy figure would hide.
- Explain why post-hoc explanations are, at best, supporting evidence in an ISO/PAS 8800 assurance argument.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from scipy.ndimage import binary_dilation

rng = np.random.default_rng(42)
torch.manual_seed(42)
torch.set_num_threads(1)
plt.rcParams.update({"figure.dpi": 100})
OI = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
pd.set_option("display.width", 160)

## 1. Why explainability matters for AV safety, and what it is not

### The problem
Deep perception networks have millions of parameters and highly non-linear mappings; the path from pixels to a detection cannot be traced by inspection. When an engineer, an assessor or an accident investigator asks "why did the system brake here?" or "why was this pedestrian missed?", the network output alone ("score 0.87") does not answer the question.

### Where explanations help
1. **Debugging and safety analysis:** finding *why* a detector fails, e.g. reliance on spurious features (shortcut learning), which is an input to the safety analysis of AI errors in ISO/PAS 8800 and to the search for triggering conditions in ISO 21448.
2. **Dataset work:** revealing biased or leaking features in the data.
3. **Incident investigation and liability:** reconstructing what the perception system relied on.
4. **Communication and trust:** for developers, assessors and, carefully, users.

### What the standards and regulations actually say (as of October 2026)
- **ISO/IEC TR 5469:2024** is an informative Technical Report. It discusses explainability and transparency among properties relevant to AI in functional safety; as a TR it cannot impose requirements.
- **ISO/PAS 8800:2024** addresses AI safety in road vehicles through safety requirements, data, V&V, safety analysis and an assurance argument. Explanation methods can support several of these activities, but post-hoc XAI is not a stand-alone acceptance criterion.
- **EU AI Act:** vehicle AI is high-risk via Article 6(1) and Annex I Section B, but its requirements reach vehicles indirectly through type-approval delegated acts (Article 2(2)); see [01](01_ai_safety_standards.ipynb). The individual right to explanation (Article 86) concerns decisions by deployers of Annex III systems, not vehicle perception.
- **NHTSA Standing General Order 2021-01** requires reporting of crashes involving ADS and Level 2 ADAS within set deadlines (third amendment effective June 2025); it does not require model explanations.
- No AV type-approval regulation that we are aware of mandates post-hoc XAI.

**Key point:** XAI is a valuable engineering and investigation tool with known reliability limits. It is not a legal requirement for AV approval and not proof of correctness.

### Example: the Uber ATG crash in Tempe, Arizona (2018)
According to the NTSB (HAR-19/03), the automated driving system first detected the pedestrian **5.6 s** before impact. Its classification alternated between vehicle, bicycle and "other", and the system **never classified her as a pedestrian**; with each reclassification the object's tracking history was not used for path prediction. A one-second "action suppression" after hazard detection, the deactivated vehicle AEB, and the operator's inattention and safety-management deficiencies were central findings. Explanations of individual classifications might have helped engineers to find the misclassification pattern earlier, but the root causes were design and safety-management failures, which XAI would not have replaced.

## 2. A small, transparent test bed

To study explanation methods we need a case where we *know* the right answer. We generate 32 × 32 grey-scale "scenes" containing a **pedestrian-like** shape (upright body with a head), a **vehicle-like** shape (wide box with wheels), or **no object** (background only), plus sensor noise and clutter, and keep the ground-truth object mask.

The background class matters. A classifier trained only on "pedestrian vs vehicle" can learn that *pedestrian = absence of vehicle features*; it then rates an empty road as a confident pedestrian, and every explanation method will (correctly) point at the background. This closed-world assumption is a classic functional insufficiency of perception classifiers.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
IMG = 32
yy, xx = np.mgrid[:IMG, :IMG]

def make_scene(cls, r, watermark=False):
    img = 0.2 + 0.15 * r.standard_normal((IMG, IMG))
    for _ in range(2):                                            # clutter blobs
        x0, y0 = r.integers(0, IMG - 4, 2)
        img[y0:y0 + r.integers(2, 4), x0:x0 + r.integers(2, 4)] += 0.4
    mask = np.zeros((IMG, IMG), bool)
    if cls == 1:                                                  # pedestrian: body + head
        h, w = r.integers(12, 17), r.integers(3, 5)
        x0, y0 = r.integers(4, IMG - 4 - w), r.integers(6, IMG - h - 1)
        mask[y0:y0 + h, x0:x0 + w] = True
        mask |= (yy - (y0 - 3)) ** 2 + (xx - (x0 + w // 2)) ** 2 <= 4
    elif cls == 0:                                                # vehicle: body + two wheels
        w, h = r.integers(13, 19), r.integers(5, 8)
        x0, y0 = r.integers(1, IMG - w - 1), r.integers(4, IMG - h - 3)
        mask[y0:y0 + h, x0:x0 + w] = True
        mask[y0 + h:y0 + h + 2, x0 + 1:x0 + 4] = True
        mask[y0 + h:y0 + h + 2, x0 + w - 4:x0 + w - 1] = True
    img[mask] += 0.25                                             # low contrast: a realistic, not trivial, task
    if watermark:
        img[0:3, 0:3] += 1.5                                      # a spurious marker in the corner
    return img.astype(np.float32), mask

def make_set(n, seed, p_mark_ped=0.0, p_mark_veh=0.0):
    r = np.random.default_rng(seed)
    y = r.integers(0, 3, n)                                       # 0 vehicle, 1 pedestrian, 2 background
    imgs, masks = zip(*[make_scene(c, r, watermark=r.random() < (p_mark_ped if c == 1 else p_mark_veh)) for c in y])
    return torch.tensor(np.stack(imgs))[:, None], torch.tensor(y), np.stack(masks)

X_tr, y_tr, _ = make_set(2000, seed=1)
X_te, y_te, M_te = make_set(400, seed=2)
CLASSES = ["vehicle", "pedestrian", "background"]

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(X_te[k, 0], cmap="gray"); ax.contour(M_te[k], levels=[0.5], colors=[OI[0]], linewidths=1)
    ax.set_title(CLASSES[int(y_te[k])]); ax.axis("off")
plt.suptitle("Synthetic scenes (orange contour = ground-truth object mask)", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Training images {len(X_tr)}, test images {len(X_te)}, class counts {np.bincount(y_tr.numpy()).tolist()} ({', '.join(CLASSES)})")

In [ ]:
class SmallCNN(nn.Module):
    """Three conv blocks; the last conv layer (8 x 8 feature maps) is the Grad-CAM target."""
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 16, 3, padding=1)
        self.c2 = nn.Conv2d(16, 32, 3, padding=1)
        self.c3 = nn.Conv2d(32, 32, 3, padding=1)
        self.fc = nn.Linear(32, 3)
    def forward(self, x, return_features=False):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)       # 16 x 16
        x = F.max_pool2d(F.relu(self.c2(x)), 2)       # 8 x 8
        feats = F.relu(self.c3(x))                    # 8 x 8 feature maps (no in-place ops)
        logits = self.fc(feats.mean(dim=(2, 3)))      # global average pooling
        return (logits, feats) if return_features else logits

def train_cnn(X, y, epochs=5, seed=0):
    torch.manual_seed(seed)
    net = SmallCNN()
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        net.train()
        perm = torch.randperm(len(X), generator=g)
        for i in range(0, len(X), 64):
            idx = perm[i:i + 64]
            opt.zero_grad(); F.cross_entropy(net(X[idx]), y[idx]).backward(); opt.step()
    return net.eval()

@torch.no_grad()
def accuracy(net, X, y):
    return (net(X).argmax(1) == y).float().mean().item()

t0 = time.perf_counter()
model = train_cnn(X_tr, y_tr)
print(f"Trained in {time.perf_counter() - t0:.1f} s; test accuracy {accuracy(model, X_te, y_te):.3f}")

## 3. Five explanation methods, implemented from scratch

We explain the **log-odds** of the target class, $f_c(x) = \log\frac{P(c\mid x)}{1-P(c\mid x)} = z_c - \log\sum_{k\ne c} e^{z_k}$. Unlike a single logit, it cannot change without the prediction changing. All perturbation-based methods and baselines use the **background level** (0.2) rather than a black image, which would itself be far outside the data.

| Method | What it computes | Resolution | Notes |
|---|---|---|---|
| **Gradient saliency** (Simonyan et al., 2014) | $\lvert\partial f_c/\partial x\rvert$, max over channels | Pixel | Local sensitivity, not contribution; noisy; reflects the model's gradient, not causation |
| **Grad-CAM** (Selvaraju et al., 2017) | $\mathrm{ReLU}\big(\sum_k \alpha_k A^k\big)$ with $\alpha_k = $ spatial mean of $\partial f_c/\partial A^k$ | Feature map (here 8 × 8), upsampled | Class-discriminative and less noisy, but **coarse**: lower resolution than gradients. Guided Grad-CAM multiplies it with guided backpropagation for detail |
| **Integrated gradients** (Sundararajan et al., 2017) | $(x - x')\odot\int_0^1 \partial f_c(x' + \alpha(x-x'))/\partial x\, d\alpha$ | Pixel | **Completeness:** attributions sum to $f_c(x) - f_c(x')$ for baseline $x'$ |
| **Occlusion** (Zeiler & Fergus, 2014) | Drop in $f_c$ when a patch is replaced by background | Patch | Perturbation-based, model-agnostic, costly |
| **Shapley values** (Lundberg & Lee, 2017) | Average marginal contribution of each patch over all orderings | Patch (superpixel) | **Local accuracy:** $f(x) = \phi_0 + \sum_i \phi_i$, with $\phi_0$ the value with all patches "absent" |

**LIME** (Ribeiro et al., 2016), not implemented here, fits a sparse linear surrogate $g$ to the model around $x$ by minimising $\sum_z \pi_x(z)\,(f(z) - g(z'))^2 + \Omega(g)$, with a proximity kernel $\pi_x(z) = \exp(-D(x,z)^2/\sigma^2)$ and a complexity penalty $\Omega$. Its explanations depend on the kernel width and the sampling, and can be unstable between runs.

In [ ]:
BACKGROUND = 0.2

def margin(logits, c):
    """Log-odds of class c: log P(c) - log(1 - P(c)) = z_c - logsumexp(z_other)."""
    other = torch.cat([logits[:, :c], logits[:, c + 1:]], dim=1)
    return logits[:, c] - torch.logsumexp(other, dim=1)

def saliency(net, x, c):
    x = x.clone().detach().requires_grad_(True)
    net.zero_grad(set_to_none=True)
    margin(net(x), c).sum().backward()
    return x.grad.abs().amax(dim=1).detach()                       # (N, H, W)

def grad_cam(net, x, c):
    logits, feats = net(x, return_features=True)
    grads = torch.autograd.grad(margin(logits, c).sum(), feats)[0] # d f_c / d A^k
    alpha = grads.mean(dim=(2, 3), keepdim=True)                   # GAP of gradients
    cam = F.relu((alpha * feats).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
    return cam[:, 0].detach()

def integrated_gradients(net, x, c, baseline=None, steps=64):
    baseline = torch.full_like(x, BACKGROUND) if baseline is None else baseline
    alphas = (torch.arange(steps, dtype=torch.float32) + 0.5) / steps    # midpoint rule
    total = torch.zeros_like(x)
    for a in alphas:
        xi = (baseline + a * (x - baseline)).requires_grad_(True)
        g = torch.autograd.grad(margin(net(xi), c).sum(), xi)[0]
        total += g
    return ((x - baseline) * total / steps)[:, 0].detach()           # signed, (N, H, W)

@torch.no_grad()
def occlusion(net, x, c, patch=6, stride=2, fill=BACKGROUND):
    base = margin(net(x), c)
    heat = torch.zeros(x.shape[0], IMG, IMG); count = torch.zeros(IMG, IMG)
    for i in range(0, IMG - patch + 1, stride):
        for j in range(0, IMG - patch + 1, stride):
            xo = x.clone(); xo[:, :, i:i + patch, j:j + patch] = fill
            drop = base - margin(net(xo), c)
            heat[:, i:i + patch, j:j + patch] += drop[:, None, None]; count[i:i + patch, j:j + patch] += 1
    return heat / count

@torch.no_grad()
def shapley_patches(net, x, c, grid=4, n_perm=300, fill=BACKGROUND, seed=0):
    """Permutation-sampling Shapley values of grid x grid patches for ONE image x (1, 1, H, W).
    Absent patches are replaced by the background value `fill`. Each permutation's marginal
    contributions sum exactly to f(x) - f(empty), so efficiency holds exactly."""
    r = np.random.default_rng(seed); n = grid * grid; s = IMG // grid
    def compose(present):                                            # present: (B, n) bool
        B = present.shape[0]
        out = torch.full((B, 1, IMG, IMG), fill)
        for p in range(n):
            i, j = divmod(p, grid)
            sel = torch.as_tensor(present[:, p])
            out[sel, :, i*s:(i+1)*s, j*s:(j+1)*s] = x[0, :, i*s:(i+1)*s, j*s:(j+1)*s]
        return out
    phi = np.zeros(n)
    for _ in range(n_perm):
        order = r.permutation(n)
        present = np.zeros((n + 1, n), bool)
        for k in range(1, n + 1):
            present[k] = present[k - 1]; present[k, order[k - 1]] = True
        v = margin(net(compose(present)), c).numpy()
        phi[order] += np.diff(v)
    phi /= n_perm
    f_empty = margin(net(compose(np.zeros((1, n), bool))), c)[0].item()
    f_full = margin(net(x), c)[0].item()
    return phi.reshape(grid, grid), f_empty, f_full

In [ ]:
# Explain one correctly classified pedestrian
with torch.no_grad():
    pred = model(X_te).argmax(1)
k = int(np.where((y_te == 1).numpy() & (pred == 1).numpy())[0][0])
x = X_te[k:k + 1]; c = 1
maps = {
    "Gradient saliency": saliency(model, x, c)[0],
    "Grad-CAM (8x8, upsampled)": grad_cam(model, x, c)[0],
    "Integrated gradients": integrated_gradients(model, x, c)[0],
    "Occlusion (drop in log-odds)": occlusion(model, x, c)[0],
}
phi, f_empty, f_full = shapley_patches(model, x, c)

fig, axes = plt.subplots(1, 6, figsize=(21, 4))
axes[0].imshow(x[0, 0], cmap="gray"); axes[0].contour(M_te[k], levels=[0.5], colors=[OI[0]])
p_ped = F.softmax(model(x), -1)[0, 1].item()
axes[0].set_title(f"Input, P(pedestrian) = {p_ped:.3f}")
for ax, (name, m) in zip(axes[1:5], maps.items()):
    m = m.numpy()
    lim = np.abs(m).max()
    ax.imshow(m, cmap="RdBu_r" if m.min() < 0 else "viridis", vmin=-lim if m.min() < 0 else 0, vmax=lim)
    ax.contour(M_te[k], levels=[0.5], colors=[OI[0]], linewidths=1); ax.set_title(name)
lim = np.abs(phi).max()
axes[5].imshow(np.kron(phi, np.ones((IMG // 4, IMG // 4))), cmap="RdBu_r", vmin=-lim, vmax=lim)
axes[5].contour(M_te[k], levels=[0.5], colors=[OI[0]], linewidths=1); axes[5].set_title("Shapley values (4x4 patches)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

ig = maps["Integrated gradients"]
with torch.no_grad():
    f_x, f_0 = margin(model(x), c)[0].item(), margin(model(torch.full_like(x, BACKGROUND)), c)[0].item()
print("Integrated gradients completeness check (log-odds of 'pedestrian', baseline = background image):")
print(f"  sum of attributions = {ig.sum().item():.4f};  f(x) - f(baseline) = {f_x - f_0:.4f}  "
      f"(relative error {abs(ig.sum().item() - (f_x - f_0)) / abs(f_x - f_0):.2%}, from the 64-step integral)")
print("Shapley local accuracy check (log-odds of 'pedestrian', absent patches = background):")
print(f"  phi_0 = f(all patches absent) = {f_empty:.4f};  sum phi_i = {phi.sum():.4f};  "
      f"phi_0 + sum phi_i = {f_empty + phi.sum():.4f};  f(x) = {f_full:.4f}")
top = np.argsort(-phi.ravel())[:3]
print("  Largest patch contributions: " + ", ".join(f"patch (row {t // 4}, col {t % 4}) {phi.ravel()[t]:+.3f}" for t in top))

Note on the Shapley "base value" $\phi_0$: here it is the model output with all patches replaced by background. With SHAP's `KernelExplainer`, $\phi_0$ is the *expected* output over a background data set. Either way, attributions sum to $f(x) - \phi_0$, **not** to $f(x)$. A statement such as "0.87 = 0.3 (shape) + 0.4 (colour) + 0.17 (context)" omits the base value; and for images, the "features" are pixels or superpixels, not abstract concepts like shape or colour.

## 4. Do the explanations point at the object? A localisation check

Because the object masks are known, we can measure the share of positive attribution inside the mask and compare it with the share of the image the mask covers (what a random map would achieve).

In [ ]:
idx = np.where((y_te == 1).numpy() & (pred == 1).numpy())[0][:60]
Xs = X_te[idx]; Ms = M_te[idx]
batch_maps = {
    "Gradient saliency": saliency(model, Xs, 1),
    "Grad-CAM": grad_cam(model, Xs, 1),
    "Integrated gradients": integrated_gradients(model, Xs, 1).clamp_min(0),
    "Occlusion": occlusion(model, Xs, 1).clamp_min(0),
}
area = Ms.mean(axis=(1, 2))
Md = np.stack([binary_dilation(mk, iterations=4) for mk in Ms])          # mask grown by one feature-map cell (4 px)
area_d = Md.mean(axis=(1, 2))
share = lambda m, msk: (m.numpy() * msk).sum(axis=(1, 2)) / (m.numpy().sum(axis=(1, 2)) + 1e-12)
loc_df = pd.DataFrame({name: [share(m, Ms).mean(), (share(m, Ms) / area).mean(),
                              share(m, Md).mean(), (share(m, Md) / area_d).mean()]
                       for name, m in batch_maps.items()},
                      index=["share inside mask", "ratio to mask area (random = 1)",
                             "share inside mask + 4 px", "ratio, mask + 4 px"]).T
print(f"Mean mask area: {area.mean():.1%} of the image; grown by 4 px: {area_d.mean():.1%} ({len(idx)} pedestrian images)")
print(loc_df.round(3).to_string())
best = loc_df.iloc[:, 0].idxmax()
print(f"\nMost concentrated on the object: {best}.")
for name, r in loc_df.iterrows():
    print(f"  {name:22s}: {r.iloc[1]:.1f}x random inside the mask, {r.iloc[3]:.1f}x random inside the mask grown by 4 px")

Grad-CAM is computed on 8 × 8 feature maps, so one cell covers 4 × 4 pixels, while the pedestrian body is only 3–4 pixels wide: after upsampling, much of its mass necessarily falls just outside the exact mask. The second pair of columns (mask grown by one feature-map cell) makes this resolution effect visible.

Concentration on the object is *plausibility*, not *faithfulness*: a map can look right and still not reflect what the model uses. That is what the next check targets.

## 5. Sanity check: model-parameter randomisation (Adebayo et al., 2018)

If an explanation method is faithful to the model, then destroying the model's learned weights must change the explanation. In the **cascading randomisation** test, layers are re-initialised from the top (output) down; after each step we compare the new maps with the original ones by Spearman rank correlation (of absolute values). A method whose maps stay similar for a randomised model is explaining the *input* (e.g. edges), not the model. Adebayo et al. found that some popular methods, e.g. guided backpropagation, largely fail this test, while plain gradients and Grad-CAM pass it.

In [ ]:
import copy

def explain_all(net, X):
    return {"Gradient saliency": saliency(net, X, 1).abs(),
            "Grad-CAM": grad_cam(net, X, 1).abs(),
            "Integrated gradients": integrated_gradients(net, X, 1, steps=32).abs()}

def rank_corr(a, b):
    """Spearman correlation; a constant map (e.g. an all-zero Grad-CAM) carries no ranking, so count it as 0."""
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return 0.0
    return spearmanr(a.ravel(), b.ravel())[0]

def mean_spearman(A_, B_):
    return float(np.mean([rank_corr(a, b) for a, b in zip(A_.numpy(), B_.numpy())]))

Xs40 = Xs[:40]
reference = explain_all(model, Xs40)
stages = ["fc", "c3", "c2", "c1"]
rand_net = copy.deepcopy(model)
torch.manual_seed(123)
rows = [{"randomised up to": "none (trained)", **{k: 1.0 for k in reference}}]
for layer in stages:
    getattr(rand_net, layer).reset_parameters()                    # cascading: keep earlier randomisations
    maps_r = explain_all(rand_net, Xs40)
    rows.append({"randomised up to": layer, **{k: mean_spearman(reference[k], maps_r[k]) for k in reference}})
sanity = pd.DataFrame(rows).set_index("randomised up to")
print("Mean Spearman rank correlation between maps of the trained and the randomised model")
print(sanity.round(3).to_string())

fig, axes = plt.subplots(3, len(stages) + 1, figsize=(15, 9))
img_k = 0
net_c = copy.deepcopy(model); torch.manual_seed(123)
for col, layer in enumerate(["none"] + stages):
    if layer != "none":
        getattr(net_c, layer).reset_parameters()
    mm = explain_all(net_c, Xs40[img_k:img_k + 1])
    for row, name in enumerate(mm):
        axes[row, col].imshow(mm[name][0], cmap="viridis"); axes[row, col].axis("off")
        if col == 0: axes[row, col].set_title(f"{name}\ntrained model", fontsize=9)
        else: axes[row, col].set_title(f"{name}\nrandomised to {layer}", fontsize=9)
plt.suptitle("Cascading model randomisation: faithful explanations should degrade", fontweight="bold")
plt.tight_layout(); plt.show()

final = sanity.iloc[-1]
for name, v in final.items():
    verdict = ("maps change substantially (passes)" if abs(v) < 0.3 else
               "maps partly persist: partly driven by the input itself" if abs(v) < 0.6 else
               "WARNING: maps remain similar to the trained model's (fails)")
    print(f"{name:22s}: correlation after full randomisation {v:+.2f} -> {verdict}")

Even a method that "passes" can retain some correlation after randomisation, because the input image itself (bright object on a dark background) shapes the gradients of any convolutional network. Integrated gradients multiplies gradients by $(x - x')$, so its maps inherit structure from the input image, which is why it keeps more correlation than plain gradients here; Adebayo et al. report the same tendency for gradient ⊙ input. A constant map (e.g. Grad-CAM that becomes all zero after the output layer is randomised) is counted as correlation 0. Adebayo et al. also propose a **data randomisation test** (retrain on permuted labels); both tests are necessary conditions for faithfulness, not proofs of it.

## 6. Using XAI in safety analysis: uncovering a shortcut

Suppose a data-collection artefact correlates with the class: in the training data, 95 % of pedestrian images carry a small bright marker in the top-left corner (think of a timestamp overlay used only on one recording campaign), and 5 % of the other images do. A network trained on such data may learn the marker instead of the pedestrian.

Accuracy on a test set **with the same artefact** looks excellent. Explanations reveal the problem, and a targeted test confirms it.

In [ ]:
X_trs, y_trs, _ = make_set(2000, seed=1, p_mark_ped=0.95, p_mark_veh=0.05)
shortcut_model = train_cnn(X_trs, y_trs, seed=0)

X_same, y_same, _ = make_set(400, seed=3, p_mark_ped=0.95, p_mark_veh=0.05)   # same artefact as training
X_clean, y_clean, M_clean = make_set(400, seed=4)                             # no marker at all
X_swap, y_swap, _ = make_set(400, seed=5, p_mark_ped=0.0, p_mark_veh=1.0)    # marker only on non-pedestrians

acc = pd.DataFrame({
    "clean model": [accuracy(model, X_same, y_same), accuracy(model, X_clean, y_clean), accuracy(model, X_swap, y_swap)],
    "shortcut model": [accuracy(shortcut_model, X_same, y_same), accuracy(shortcut_model, X_clean, y_clean),
                       accuracy(shortcut_model, X_swap, y_swap)]},
    index=["test with same artefact", "clean test (no marker)", "marker only on non-pedestrians"])
print("Test accuracy"); print(acc.round(3).to_string())

corner = np.zeros((IMG, IMG), bool); corner[0:4, 0:4] = True
ped = (y_same == 1).numpy()
attr = {name: integrated_gradients(net, X_same[ped][:60], 1).clamp_min(0).numpy()
        for name, net in [("clean model", model), ("shortcut model", shortcut_model)]}
share = {name: float(((a * corner).sum(axis=(1, 2)) / (a.sum(axis=(1, 2)) + 1e-12)).mean()) for name, a in attr.items()}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
j = 0
axes[0].imshow(X_same[ped][j, 0], cmap="gray"); axes[0].set_title("Pedestrian image with marker")
for ax, name in zip(axes[1:], attr):
    ax.imshow(attr[name][j], cmap="viridis"); ax.set_title(f"{name}: IG (positive part)\ncorner share {share[name]:.1%}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

print(f"\nShare of positive IG attribution in the 4x4 corner (1.6 % of the image): "
      f"clean model {share['clean model']:.1%}, shortcut model {share['shortcut model']:.1%}")
drop = acc.loc["test with same artefact", "shortcut model"] - acc.loc["marker only on non-pedestrians", "shortcut model"]
print(f"Shortcut model: accuracy falls by {drop:.1%} when the marker no longer correlates with the class.")

In ISO/PAS 8800 terms, the explanation pointed to a **data-related insufficiency** (a spurious correlation in the dataset) that caused an **AI error** under a specific condition. The *evidence* that the problem exists, and later that it is fixed, comes from the targeted test sets, not from the heat map itself. That is the appropriate role of XAI: a hypothesis generator for safety analysis, followed by verification.

## 7. Limits of XAI as safety evidence (ISO/PAS 8800 perspective)

| Limitation | Consequence for an assurance argument |
|---|---|
| **Unfaithfulness:** maps may reflect the input or the method more than the model (Adebayo et al., 2018) | Any explanation method used as evidence needs its own sanity checks and faithfulness metrics |
| **Manipulability:** explanations can be changed by imperceptible input perturbations or by adversarially fine-tuned models while predictions stay the same (Dombrowski et al., 2019) | Explanations are not robust evidence under adversarial conditions; relevant for ISO/SAE 21434 as well |
| **Method dependence:** different methods (and baselines, kernel widths, patch sizes) give different answers | Fix and justify the method and its parameters; report variability |
| **Confirmation bias:** a plausible-looking map invites over-trust | Use explanations to generate hypotheses, then test them with data |
| **Local, not global:** one explanation covers one input | Statements about the ODD need aggregated, statistical evidence per slice |
| **Detectors and fusion:** most methods were designed for classifiers; for detectors the target is a box's class logit, and fused or temporal models complicate attribution | Explanations for production perception stacks are research topics, not mature tools |
| **Attention is not explanation:** attention weights show information routing, not necessarily causal importance (Jain & Wallace, 2019; Wiegreffe & Pinter, 2019) | Use gradient-weighted attention or rollout with care |

**Bottom line:** in an ISO/PAS 8800 assurance argument, explanations can support claims about the *absence of known shortcut features* or the *plausibility of learned features*, provided the method passed sanity checks and the claim is confirmed by targeted tests. They do not replace statistical V&V, safety analysis or field monitoring.

## 8. Documenting explanations for an AI component

The template below is **illustrative**. All numbers are placeholders, not results, and acceptance criteria come from the project's AI safety requirements and SOTIF validation targets, not from ASIL values (ISO 26262 ASILs do not define perception error rates).

In [ ]:
documentation_template = """
================================================================
AI COMPONENT DOCUMENTATION - EXPLAINABILITY SECTION
(illustrative; informed by ISO/PAS 8800:2024 and ISO/IEC TR 5469:2024)
================================================================
COMPONENT : Pedestrian classifier, camera channel      VERSION: <x.y.z>
AUTHOR    : <name>                                     DATE   : <date>

1. EXPLANATION METHODS AND PARAMETERS
   Primary  : Integrated gradients (target: log-odds; baseline: background image, 64 steps)
   Secondary: Grad-CAM on last conv block (8x8, bilinear upsampling)
   Sanity checks: model-randomisation test, Spearman correlation
                  after full randomisation: IG <value>, Grad-CAM <value>

2. INTENDED USE OF EXPLANATIONS
   - Safety analysis: search for shortcut features per ODD slice
   - Incident investigation: reconstruction of attribution for logged frames
   - NOT used as acceptance evidence on their own

3. FINDINGS (example)
   F-01: Attribution concentrated on corner marker in campaign C-07 data
         -> dataset insufficiency; targeted test set T-SC-01 created
         -> accuracy with marker decorrelated: <value> (before fix) / <value> (after fix)

4. ACCEPTANCE CRITERIA (from AI safety requirements, per ODD slice)
   FNR <= <X> in slice <Y> with 95 % confidence  -> status: <met / not met>

5. LIMITATIONS
   Explanations are local, method-dependent and can be manipulated;
   see Section 7 of the teaching notebook.
================================================================
"""
print(documentation_template)

## 9. Accuracy vs interpretability

- **Inherently interpretable models** (linear models, small trees, prototype networks) are easier to explain but often insufficient for camera or LiDAR perception.
- **Complex models with post-hoc explanations** keep performance, but the explanations themselves must be validated.
- **Architectural safety measures** often matter more than either: an interpretable *monitor* or rule-based plausibility check next to a complex detector (e.g. a LiDAR free-space check that can veto a "no obstacle" output) gives a safety argument that does not rely on explaining the network.

Public evidence on how AV developers use XAI in production is limited; published safety reports focus on safety cases, scenario-based testing and field performance rather than on XAI.

## Key takeaways

- XAI supports debugging, safety analysis and investigation; **no AV regulation mandates post-hoc XAI**, and ISO/IEC TR 5469 is informative.
- Know what each method computes: saliency = local gradient; Grad-CAM = coarse, class-discriminative feature-map weighting; integrated gradients = attributions summing to $f(x) - f(x')$; Shapley values sum to $f(x) - \phi_0$, **not to $f(x)$**.
- **Sanity checks** (model and data randomisation) and **localisation/faithfulness checks** are prerequisites before explanations are used as evidence.
- XAI is most useful as a **hypothesis generator** (e.g. a shortcut feature) whose findings are then confirmed with targeted tests.
- In an ISO/PAS 8800 assurance argument, explanations are supporting evidence at best.

## Self-assessment

1. Why does Grad-CAM have lower spatial resolution than gradient saliency?
2. A Shapley explanation lists contributions +0.30, +0.15 and +0.07 for a detection score of 0.87. What is the base value?
3. What does the model-randomisation test check, and what does passing it not prove?
4. How can XAI contribute to the safety analysis of AI errors in ISO/PAS 8800?
5. Why is "the explanation satisfies ISO/IEC TR 5469" not a meaningful compliance statement?

<details>
<summary>Answers</summary>

1. Grad-CAM weights the feature maps of a chosen convolutional layer, whose spatial size is a fraction of the input (8 × 8 here, often 7 × 7 or 14 × 14 in large CNNs); the map is then upsampled. Gradient saliency is computed per input pixel.
2. By local accuracy $f(x) = \phi_0 + \sum\phi_i$: $\phi_0 = 0.87 - 0.52 = 0.35$.
3. It checks that explanations depend on the learned parameters: if maps barely change when the weights are randomised, the method is not explaining the model. Passing is a necessary, not a sufficient condition for faithfulness.
4. By generating hypotheses about why errors occur (spurious features, reliance on context, sensitivity to particular conditions), which are then confirmed with targeted test sets and fed back into dataset requirements, architecture or ODD restrictions.
5. A Technical Report is informative and contains no requirements to comply with; compliance statements need a normative reference and an assessment of the full set of work products.
</details>

## References

- Selvaraju, R. R. et al. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. ICCV 2017. https://doi.org/10.1109/ICCV.2017.74
- Simonyan, K., Vedaldi, A., Zisserman, A. (2014). Deep inside convolutional networks: visualising image classification models and saliency maps. ICLR Workshop 2014. arXiv:1312.6034
- Sundararajan, M., Taly, A., Yan, Q. (2017). Axiomatic attribution for deep networks. ICML 2017. arXiv:1703.01365
- Zeiler, M. D., Fergus, R. (2014). Visualizing and understanding convolutional networks. ECCV 2014. arXiv:1311.2901
- Lundberg, S. M., Lee, S.-I. (2017). A unified approach to interpreting model predictions. NeurIPS 2017. arXiv:1705.07874
- Ribeiro, M. T., Singh, S., Guestrin, C. (2016). "Why should I trust you?" Explaining the predictions of any classifier. KDD 2016. arXiv:1602.04938
- Adebayo, J. et al. (2018). Sanity checks for saliency maps. NeurIPS 2018. arXiv:1810.03292
- Dombrowski, A.-K. et al. (2019). Explanations can be manipulated and geometry is to blame. NeurIPS 2019. arXiv:1906.07983
- Jain, S., Wallace, B. C. (2019). Attention is not explanation. NAACL 2019. arXiv:1902.10186
- Wiegreffe, S., Pinter, Y. (2019). Attention is not not explanation. EMNLP 2019. arXiv:1908.04626
- Geirhos, R. et al. (2020). Shortcut learning in deep neural networks. *Nature Machine Intelligence* 2, 665–673. https://doi.org/10.1038/s42256-020-00257-z
- NTSB (2019). Collision Between Vehicle Controlled by Developmental Automated Driving System and Pedestrian, Tempe, Arizona, March 18, 2018. Highway Accident Report NTSB/HAR-19/03. https://www.ntsb.gov/investigations/Pages/HWY18MH010.aspx
- ISO/PAS 8800:2024; ISO/IEC TR 5469:2024; Regulation (EU) 2024/1689.

---
**Navigation:** ← [05 Conformal prediction](05_conformal_prediction.ipynb) · [Module index](README.md) · [07 Statistical safety validation](07_statistical_safety_validation.ipynb) →

### Further practice

**Try this:** change the baseline of integrated gradients (black image vs blurred input) and the patch size of the Shapley explanation, and compare the localisation scores; add a data-randomisation test (retrain on shuffled labels).

The author's companion repository [AI-Perception-Safety-Workshop](https://github.com/milinpatel07/AI-Perception-Safety-Workshop) contains an XAI exercise on real images.